In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PRICE = 'Listing Price (USD)'

In [ ]:
# Read the original files
train_raw = pd.read_csv('BoatData_Train.csv')
test_raw = pd.read_csv('BoatData_NoPrice.csv')

print('Priced file:', train_raw.shape)
print('Prediction file:', test_raw.shape)
display(train_raw.head())

# Data inspection and cleaning


In [ ]:
train_raw.info()
display(train_raw.isna().sum().rename('Missing values'))
print('Exact repeated training rows:', train_raw.duplicated().sum())

train = train_raw.drop_duplicates().reset_index(drop=True).copy()
test = test_raw.copy()

## Clean numerical features

In [ ]:
### Clean Training

def clean_numeric_text(df):
    df = df.copy()

    # Draft and headroom: extract metres
    # Example: 6’ 11”2.1 m -> 2.1
    for col in ['Draft', 'Maximum headroom']:
        df[col] = (
            df[col]
            .str.replace('’', ',')
            .str.replace('”', ',')
            .str.split(',').str[-1]
            .str.replace('m', '')
            .str.strip()
            .astype(float)
        )

    # Sail areas: extract square metres
    # Example: 1152 ft²107 m² -> 107
    for col in ['Upwind sail area', 'Downwind sail area', 'Mainsail area']:
        df[col] = (
            df[col]
            .str.split('ft²').str[-1]
            .str.replace('m²', '')
            .str.strip()
            .astype(float)
        )

    df['Still in production'] = (df['Last built hull'].eq('Still in production').astype(int))

    df['Discontinued'] = (df['Last built hull'].eq('Discontinued').astype(int))

    df['Last built hull'] = pd.to_numeric(df['Last built hull'], errors = 'coerce')


    # Average of the one or two listed horsepower values
    engine_parts = (
        df['Engine(s) power']
        .str.replace('HP', '')
        .str.strip()
        .str.split('/')
    )

    df['Engine average power'] = (
        engine_parts.str[0].astype(float)
        + engine_parts.str[-1].astype(float)
    ) / 2

    return df.drop(columns = ['Engine(s) power'])


train = clean_numeric_text(train_raw.drop_duplicates().reset_index(drop = True))
test = clean_numeric_text(test_raw) 

### Correct invalid values and anomalies 


In [ ]:
# Inspect the truncated years alongside the proposed full-year values
display(train.loc[train['Year'].isin([5, 6]), ['Make', 'Variant', 'Year', 'Model']])
display(train.loc[train['Year'].isin([2005, 2006]), ['Make', 'Variant', 'Year', 'Model']].head())

# Inspect physically invalid mainsail measurements
display(train.loc[train['Mainsail area'] <= 0, ['Make', 'Variant', 'Mainsail area']])

for df in [train, test]:
    df['Year'] = df['Year'].replace({5: 2005, 6: 2006})
    df['Mainsail area'] = df['Mainsail area'].mask(df['Mainsail area'] <= 0)


### Check extreme values


In [ ]:
numeric_data = train.select_dtypes(include = 'number').drop(columns = ['Still in production', 'Discontinued'])

q1 = numeric_data.quantile(0.25)
q3 = numeric_data.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_mask = numeric_data.lt(lower) | numeric_data.gt(upper)

outlier_summary = pd.DataFrame({
    'Flagged values': outlier_mask.sum(),
    'Lower IQR bound': lower,
    'Upper IQR bound': upper
})
outlier_summary = outlier_summary.loc[outlier_summary['Flagged values'] > 0]
display(outlier_summary.sort_values('Flagged values', ascending = False).round(2))
print('Listings with at least one flag:', outlier_mask.any(axis = 1).sum())

# Display rows with at least one flagged value for inspection
has_outlier = outlier_mask.any(axis=1)

flagged_rows = train.loc[has_outlier].copy()
flagged_rows["Outlier columns"] = outlier_mask.loc[has_outlier].apply(
    lambda row: row.index[row].tolist(), # Keep only columns with outliers
    axis=1
)
display(flagged_rows.head())


### Convert categorical features

In [ ]:
# Clean categorical columns
def clean_categories(df):
    df = df.copy()

    # Clean formatting in all text columns
    text_cols = df.select_dtypes(include=['object', 'string']).columns

    for col in text_cols:
        df[col] = (
            df[col]
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
            .replace('', np.nan)
        )

    # Standardize known accented spellings in boat identifiers
    for col in ['Make', 'Model', 'Variant']:
        df[col] = (
            df[col]
            .str.replace('é', 'e')
            .str.replace('É', 'E')
        )

    # Inspect Variant inconsistencies AFTER basic cleanup


    return df

In [ ]:
train = clean_categories(train)

variant_check = (
    train.groupby(['Make', 'Model'])['Variant']
    .agg(
        n_variants='nunique',
        variants=lambda x: sorted(x.dropna().unique())
    )
)
print('Models with multiple Variant labels:')
display(variant_check.loc[variant_check['n_variants'] > 1])

variant_fixes = {
    '450s': '450S',
    'OCEANIS 55': 'Oceanis 55',
    '40 Impression': 'Impression 40',
    '45 Impression': 'Impression 45',
    '50 Impression': 'Impression 50'
}
train['Variant'] = train['Variant'].replace(variant_fixes)

## Clean test dataset

In [ ]:
# Clean test
test = clean_numeric_text(test_raw)

# Apply the same fixed year correction
test['Year'] = test['Year'].replace({
    5: 2005,
    6: 2006
})

# Apply the same invalid-measurement rule
test['Mainsail area'] = test['Mainsail area'].mask(
    test['Mainsail area'] <= 0,
    np.nan
)

test = clean_categories(test)
test['Variant'] = test['Variant'].replace(variant_fixes)

# Part 1 


## Q1
Compute the average mainsail area for Beneteau boats and identify the category with the highest median draft.

In [ ]:
average_mainsail = train.loc[train['Make'] == 'Beneteau', 'Mainsail area'].mean()
median_draft = train.groupby('Category')['Draft'].median()

print(f'Average Beneteau mainsail area: {average_mainsail:.2f} m^2')
print(f'Category with the highest median draft: {median_draft.idxmax()}')
print(f'Highest median draft: {median_draft.max():.2f} m')


## Q2

- Exact duplicate listings were removed
- text measurements were converted to numeric metric values,  
- Categorical feature aliases were standardized so equivalent labels are grouped together.

- The truncated construction years  
- Invalid mainsail areas 
- production-status (last built hull) was clean 
- IQR flags were reviewed in the context of boat dimensions, rather than treated as automatic errors.

The same fixed rules were applied to the prediction file. Missing features remain missing during exploratory analysis; model imputation is learned later from each training fold. This keeps descriptive statistics based on observed values and protects validation from preprocessing leakage.


## Q3

Compare numeric predictors, excluding price and the production-status indicators.


In [ ]:
numeric_data = train.select_dtypes(include = 'number').drop(
    columns = [PRICE, 'Still in production', 'Discontinued']
)
correlation = numeric_data.corr()
corr_pairs = correlation.unstack()
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) > corr_pairs.index.get_level_values(1)]
display(corr_pairs.dropna().sort_values(key = lambda x: x.abs(), ascending = False).head(3))

plot_pairs = [
    ('Beam (width)', 'Deck area'),
    ('Upwind sail area', 'Mainsail area'),
    ('Upwind sail area', 'Hull length')
]

fig, axes = plt.subplots(3, 1, figsize = (6, 10))
for ax, (x, y) in zip(axes, plot_pairs):
    sns.scatterplot(data = train, x = x, y = y, hue = 'Hull type', ax = ax)
plt.tight_layout()
plt.show()


- beam and deck area: wider boats have larger deck footprints. Deck area = hull length x beam, so the correlation partly reflects how the variable was constructed. deck area being a product of length and beam does not create exact linear multicollinearity. 

- Upwind sail area and mainsail area: both describe the sail plan. Hull length and upwind sail area also move together, with longer boats generally carrying more sail area. Keeping upwind sail area while omitting mainsail area is a reasonable simplification, particularly because mainsail area has more missing values.

- The hull-type colors show that boat composition contributes to these patterns, particularly for width and deck area. 


## Q4


In [ ]:
# Price distribution
train[PRICE].plot.hist(bins = 30, figsize = (7, 4))
plt.xlabel('Listing price (USD)')
plt.ylabel('Number of listings')
plt.title('Distribution of boat listing prices')
plt.tight_layout()
plt.show()

# Expensive boats: highest price decile
threshold = train[PRICE].quantile(0.90)
expensive = train.loc[train[PRICE] >= threshold]
other = train.loc[train[PRICE] < threshold]
print(f'Expensive boat threshold: ${threshold:,.0f}')
print('Number of expensive boats:', len(expensive))

In [ ]:
# Compare median numerical characteristics
numeric_cols = ['Year', 'Hull length', 'Beam (width)', 'Light displacement (MLC)']

numeric_comparison = pd.DataFrame({
    'Expensive': expensive[numeric_cols].median(),
    'Other': other[numeric_cols].median()
})

display(numeric_comparison)


# Compare category percentages within each group
for col in ['Hull type', 'Make', 'Category', 'Model']:
    comparison = pd.DataFrame({
        'Expensive (%)': expensive[col].value_counts(normalize = True) * 100,
        'Other (%)': other[col].value_counts(normalize = True) * 100
    }).fillna(0)

    most_exp = (comparison.sort_values('Expensive (%)', ascending = False).head(5))

    # Horizontal bars make long category names easier to read
    most_exp.plot.barh(figsize=(7, 4))
    plt.title(col)
    plt.xlabel('Percentage within each group')
    plt.tight_layout()
    plt.show()


    make_comparison = comparison.copy()

    make_comparison['Expensive Ratio'] = (
       expensive[col].value_counts()
        / train[col].value_counts() * 100
    ).fillna(0)

    display(
        make_comparison
        .sort_values('Expensive Ratio', ascending=False)
        .head(15)
        .round(1)
    )

- Listing prices right-skewed, with a small number of very expensive boats pulling the mean above the median. 
- Expensive boats tend to be newer, longer, wider and heavier. 
- Catamarans are much more common among expensive listings than among other boats.

Lagoon is the most common make among expensive listings. Its prominence among expensive boats is not explained solely by having more Lagoon listings (25.5% > 10%). This doesn't mean brand premium, still depending on others like catamarans, and dimensions, age and specifications may explain their higher prices. 

## Q5
Compare overall regional prices, then compare regions within hull type and inspect construction year and size.


In [ ]:
region_summary = train.groupby('Geographic Region')[PRICE].agg(['count', 'mean', 'median'])
display(region_summary.round(0))

region_summary[['mean', 'median']].plot.bar(figsize = (7, 4), rot = 0)
plt.ylabel('Listing price (USD)')
plt.title('Mean and median price by region')
plt.tight_layout()
plt.show()

groups = train.groupby(['Geographic Region', 'Hull type'])
display(groups[PRICE].agg(['count', 'mean', 'median']).round(0))

fig, axes = plt.subplots(1, 2, figsize = (12, 5), sharey = True)
sns.boxplot(data = train, x = 'Geographic Region', y = PRICE, ax = axes[0])
sns.boxplot(data = train, x = 'Geographic Region', y = PRICE, hue = 'Hull type', ax = axes[1])
axes[0].set_title('Overall regional comparison') 
axes[1].set_title('Within hull types')
plt.tight_layout()
plt.show()

# Compare the characteristics of boats listed in each group
display(groups[['Year', 'Hull length']].median().round(2))


Region is associated with listing price, but the ranking is not consistent across hull types. The Caribbean has the highest overall typical price, followed by Europe and the USA. Within catamarans, Europe has the highest median; within monohulls, the USA has the highest median.

The overall pattern partly reflects the mix of boats listed: catamarans are more common in the Caribbean and relatively uncommon in the USA. Europe has higher median prices than the Caribbean within both hull types, despite the Caribbean ranking higher overall.

USA catamarans are typically older and smaller than those in Europe, which could contribute to their lower prices. USA monohulls are also older and smaller but have higher median prices, so other factors may matter. These comparisons do not isolate a causal location effect, and the small USA catamaran sample limits confidence in that comparison.


## Q6


In [ ]:
price_by_year = train.groupby('Year')[PRICE].agg(['count', 'mean', 'median'])
display(price_by_year.round(0))

price_by_year[['mean', 'median']].plot.line(figsize = (7, 4), marker="o")
plt.xlabel('Construction year')
plt.ylabel('Listing price (USD)')
plt.title('Price by year built')
plt.tight_layout()
plt.show()

# Inspect the 2009 dip using nearby years
nearby_years = train.loc[train['Year'].isin([2008, 2009, 2010])]
display(nearby_years.groupby('Year')[['Hull length', 'Beam (width)']].median().round(2))
display(nearby_years.groupby(['Year', 'Hull type'])[PRICE].agg(['count', 'median']).round(0))


Newer boats generally have higher listing prices, particularly toward the recent end of the construction-year range. Means exceed medians, consistent with expensive listings pulling the averages upward.

The dip examined above appears partly related to sample composition: that cohort contains smaller boats and relatively few catamarans. Within hull types, monohull prices are closer to the adjacent cohort, while catamarans do not show the same dip. Small subgroup counts make detailed comparisons uncertain. 

# Part 2

In [ ]:
# Preserve missing feature values until each fold is prepared
train = train.dropna(subset = [PRICE]).reset_index(drop = True).copy()
y = train[PRICE].copy()

base_numeric_features = [
    'Year', 'Hull length', 'Beam (width)', 'Deck area', 'Draft',
    'Upwind sail area', 'Light displacement (MLC)', 'Engine average power'
]
ratio_features = [
    'Beam_length_ratio', 'Sail_area_per_displacement', 'Power_per_displacement'
]
numeric_features = base_numeric_features + ratio_features
# Model and variant are omitted as predictors to avoid many sparse labels
categorical_features = ['Make', 'Hull type', 'Geographic Region']
features = numeric_features + categorical_features


def make_features(df):
    X = df[base_numeric_features + categorical_features].copy()
    # Describe how broad the boat is
    X['Beam_length_ratio'] = X['Beam (width)'] / X['Hull length']
    # Describe sail area relative to mass
    X['Sail_area_per_displacement'] = X['Upwind sail area'] / X['Light displacement (MLC)']
    # Describe engine specification relative to mass
    X['Power_per_displacement'] = X['Engine average power'] / X['Light displacement (MLC)']
    return X[features]


## Imputation of features


In [ ]:
def impute_features(train_part, other_part):
    train_part = train_part.copy()
    other_part = other_part.copy()

    # Learn every mean from the original training portion only
    variant_means = train_part.groupby(['Make', 'Variant'])[base_numeric_features].mean()
    make_means = train_part.groupby('Make')[base_numeric_features].mean()
    overall_means = train_part[base_numeric_features].mean()

    for df in [train_part, other_part]:

        # 1. Make + Variant
        variant_fill = df[['Make', 'Variant']].join(
            variant_means, on = ['Make', 'Variant']
        )
        df[base_numeric_features] = df[base_numeric_features].fillna(
            variant_fill[base_numeric_features]
        )
        
        # 2. Make
        make_fill = df[['Make']].join(make_means, on = 'Make') # Each make only have one hull type
        df[base_numeric_features] = df[base_numeric_features].fillna(
            make_fill[base_numeric_features]
        )

        # 3. Overall training mean
        df[base_numeric_features] = df[base_numeric_features].fillna(overall_means)

        # 4. Missing categorical predictors
        df[categorical_features] = df[categorical_features].fillna('Unknown')

    return train_part, other_part


## Model comparison and validation

In [ ]:
random_cv = KFold(n_splits = 5, shuffle = True, random_state = 42)
group_cv = GroupKFold(n_splits = 5)

# Split the cleaned rows before imputation; keep Make and Variant for lookups
cv_splits = {
    'Random CV': list(random_cv.split(train)),
    'Unseen make CV': list(group_cv.split(train, y, groups = train['Make']))
}


In [ ]:
results = []

for cv_name, splits in cv_splits.items():
    for fold, (train_idx, val_idx) in enumerate(splits, start = 1):

        # 1. Split and impute using this fold's training rows
        train_fold, val_fold = impute_features(
            train.iloc[train_idx],
            train.iloc[val_idx]
        )

        # 2. Construct features and separate the target
        X_train = make_features(train_fold)
        X_val = make_features(val_fold)

        y_train = train_fold[PRICE]
        y_val = val_fold[PRICE]

        # 3. Select numerical features
        X_train_num = X_train[numeric_features]
        X_val_num = X_val[numeric_features]

        # 4. Encode categorical features
        encoder = OneHotEncoder(
            handle_unknown = 'ignore',
            min_frequency = 5,
            sparse_output = False
        )

        X_train_cat = encoder.fit_transform(X_train[categorical_features])
        X_val_cat = encoder.transform(X_val[categorical_features])

        category_names = encoder.get_feature_names_out(categorical_features)

        X_train_cat = pd.DataFrame(
            X_train_cat, columns = category_names, index=X_train.index
        )
        X_val_cat = pd.DataFrame(
            X_val_cat, columns = category_names, index=X_val.index
        )

        # 5. Combine numerical and encoded categorical features
        X_train_encoded = pd.concat([X_train_num, X_train_cat], axis = 1)
        X_val_encoded = pd.concat([X_val_num, X_val_cat], axis = 1)

        # 6. Create a version with standardized numerical features
        X_train_scaled = X_train_encoded.copy()
        X_val_scaled = X_val_encoded.copy()

        scaler = StandardScaler()

        X_train_scaled[numeric_features] = scaler.fit_transform(X_train_num)
        X_val_scaled[numeric_features] = scaler.transform(X_val_num)

        # 7. Fit models: Ridge uses scaled inputs; RF uses unscaled inputs
        ridge = Ridge(alpha = 10)
        ridge.fit(X_train_scaled, y_train)

        log_ridge = Ridge(alpha = 10)
        log_ridge.fit(X_train_scaled, np.log1p(y_train))

        rf = RandomForestRegressor(
            n_estimators = 300,
            min_samples_leaf = 3,
            random_state = 42,
            n_jobs = -1
        )
        rf.fit(X_train_encoded, y_train)

        log_rf = RandomForestRegressor(
            n_estimators = 300,
            min_samples_leaf = 3,
            random_state = 42,
            n_jobs = -1
        )
        log_rf.fit(X_train_encoded, np.log1p(y_train))

        # 8. Predict validation prices in dollars
        predictions = {
            'Median baseline': [y_train.median()] * len(y_val),
            'Ridge': ridge.predict(X_val_scaled),
            'Log-target Ridge': np.expm1(log_ridge.predict(X_val_scaled)),
            'Random forest': rf.predict(X_val_encoded),
            'Log-target RF': np.expm1(log_rf.predict(X_val_encoded))
        }

        # 9. Record validation scores
        for model_name, prediction in predictions.items():
            results.append({
                'CV': cv_name,
                'Fold': fold,
                'Model': model_name,
                'MAE': mean_absolute_error(y_val, prediction),
                'RMSE': np.sqrt(mean_squared_error(y_val, prediction)),
                'R2': r2_score(y_val, prediction)
            })

## Model evaluation

In [ ]:
results = pd.DataFrame(results)
summary = (
    results.groupby(['CV', 'Model'])
    .agg(
        MAE = ('MAE', 'mean'),
        MAE_std = ('MAE', 'std'),
        RMSE = ('RMSE', 'mean'),
        R2 = ('R2', 'mean')
    )
    .sort_values(['CV', 'MAE'])
)
display(summary.round(2))

Random forest has the highest R^2 under both validation schemes, so it is selected for the final fit. Log-target RF has lowest MAE, also valid for best model (If want to use it, just uncomment the line below rf model fitting). The median baseline is substantially less accurate.

Errors are larger and less stable when entire makes are held out. This indicates that generalizing to unfamiliar designs is harder than predicting a similar mix of listings. The results support using physical specifications, while treating estimates for unfamiliar makes with greater uncertainty. 

In [ ]:
### Feature importance on validation

In [ ]:
# encoded_importance = pd.Series(
#     rf.feature_importances_, index = X_val_encoded.columns
# )
# importance = encoded_importance[numeric_features].copy()

# # encoded_importance = pd.Series(
# #     log_ridge.coef_, index = X_val_encoded.columns
# # )

# # For encoded categories, sum up all categories' importance
# for col in categorical_features:
#     matching_columns = encoded_importance.index.str.startswith(col + '_')
#     importance[col] = encoded_importance[matching_columns].sum()

# importance = importance.sort_values(ascending = False)
# display(importance.round(3))
# importance.head(10).sort_values().plot.barh(figsize = (7, 5))
# plt.xlabel('Feature importance')
# plt.title('Final random forest feature importance')
# plt.tight_layout()
# plt.show()


## Refit model on train


In [ ]:
# 1. Impute using means learned from all priced training rows
train_full, test_imputed = impute_features(train, test)

# 2. Construct features and separate training prices
X_full = make_features(train_full)
X_test = make_features(test_imputed)

y_full = train_full[PRICE]

# 3. Select numerical features
X_full_num = X_full[numeric_features]
X_test_num = X_test[numeric_features]

# 4. Fit categorical encoding on all training rows
final_encoder = OneHotEncoder(
    handle_unknown = 'ignore',
    min_frequency = 5,
    sparse_output = False
)

X_full_cat = final_encoder.fit_transform(X_full[categorical_features])
X_test_cat = final_encoder.transform(X_test[categorical_features])

category_names = final_encoder.get_feature_names_out(categorical_features)

X_full_cat = pd.DataFrame(
    X_full_cat, columns = category_names, index = X_full.index
)
X_test_cat = pd.DataFrame(
    X_test_cat, columns = category_names, index = X_test.index
)

# 5. Combine numerical and encoded categorical features
X_full_encoded = pd.concat([X_full_num, X_full_cat], axis = 1)
X_test_encoded = pd.concat([X_test_num, X_test_cat], axis = 1)

# 6. Refit the selected random forest on all training rows
final_model = RandomForestRegressor(
    n_estimators = 300,
    min_samples_leaf = 3,
    random_state = 42,
    n_jobs = -1
)

# final_model.fit(X_full_encoded, y_full)
# if pick log target RF: 
final_model.fit(X_full_encoded, np.log1p(y_full))

## Feature importance of final model


In [ ]:
# # Extract linear regression coefficients
# coef = pd.Series(
#     final_model.coef_, index=X_full_encoded.columns
# )

# importance = pd.Series(dtype=float)

# # Numeric features: |coefficient| * std(X)
# for col in numeric_features:
#     importance[col] = abs(coef[col]) * X_full_encoded[col].std(ddof=0)

# # Categorical features: std of combined category contribution
# for col in categorical_features:
#     matching_columns = coef.index[coef.index.str.startswith(col + '_')]
#     contribution = X_full_encoded[matching_columns] @ coef[matching_columns]
#     importance[col] = contribution.std(ddof=0)

# importance = importance.sort_values(ascending=False)

In [ ]:
encoded_importance = pd.Series(
    final_model.feature_importances_, index = X_full_encoded.columns
)
importance = encoded_importance[numeric_features].copy()

# For encoded categories, sum up all categories' importance
for col in categorical_features:
    matching_columns = encoded_importance.index.str.startswith(col + '_')
    importance[col] = encoded_importance[matching_columns].sum()

importance = importance.sort_values(ascending = False)
display(importance.round(3))
importance.head(10).sort_values().plot.barh(figsize = (7, 5))
plt.xlabel('Feature importance')
plt.title('Final random forest feature importance')
plt.tight_layout()
plt.show()


Light displacement and deck area dominate the fitted model, indicating that size-related information is central to its predictions. Construction year and make provide additional information, while engine specifications and the constructed ratios receive smaller importance scores.

Hull type receives little direct importance because width, deck area and make contain overlapping information. In this sample, the observed width ranges of monohulls and catamarans do not overlap. The low score therefore does not contradict the earlier price differences between hull types.

Correlated predictors can share or exchange split credit when the model is refitted. Numerical features with many possible split points can also be favored by this importance measure.


## Out-of-sampel prediction


In [ ]:
test_pred = final_model.predict(X_test_encoded)
# If using log-target RF: 
# test_pred = np.expm1(final_model.predict(X_test_encoded))
prediction_results = test[['Make', 'Variant', 'Year', 'Geographic Region', 'Hull type', 'Light displacement (MLC)']].copy()
prediction_results['Predicted Price (USD)'] = test_pred

display(prediction_results.round(0))
print(f'Mean predicted price: ${test_pred.mean():,.0f}')
print(f'Median predicted price: ${np.median(test_pred):,.0f}')
